# Zero-shot detector from class descriptions
Pretrained NLI weights; no prompt-injection training. Run after the comparison notebook.

In [1]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"

import platform
import random
import re
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer

OUTPUT_DIR = Path.cwd()
RANDOM_SEED = 15
MODEL_NAME = "microsoft/deberta-v3-base"
MODEL_REVISION = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
MAX_TOKENS = 128
BATCH_SIZE = 8
INFERENCE_BATCH_SIZE = 32
VALIDATION_FPR_LIMIT = 0.05
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))


hardware = platform.machine() + f" CPU; {os.cpu_count()} logical cores"
if DEVICE == "cuda":
    hardware = torch.cuda.get_device_name(0)
print("Python:", platform.python_version())
print("Hardware:", hardware)

Python: 3.13.16
Hardware: NVIDIA L4


In [2]:
%run ./data_preparation.ipynb

Loaded rows: 1662


,dataset,rows,source_columns,negative,positive,positive_pct,missing_target,missing_text,blank_text,exact_excess_text,exact_excess_text_label,normalized_excess
0,deepset,662,2,399,263,39.728097,0,0,0,0,0,0
1,safeguard,1000,2,695,305,30.500000,0,0,0,3,3,3


Entity and timestamp checks: unavailable; source files contain only text and label.


,dataset,near_pairs,nonexact_near_pairs,rows_in_near_pairs
0,deepset,28,28,49
1,safeguard,17,14,24


,dataset,split,removed
0,deepset,train,11
1,deepset,validation,1
2,safeguard,test,1
3,safeguard,train,9


,dataset,split,rows,positive,negative,positive_pct
0,deepset,test,116,60,56,51.724138
1,deepset,train,425,158,267,37.176471
2,deepset,validation,109,40,69,36.697248
3,safeguard,test,199,63,136,31.658291
4,safeguard,train,631,192,439,30.427892
5,safeguard,validation,160,48,112,30.000000


Training overlap counts: {'train_exact_test': 0, 'train_normalized_test': 0, 'train_near_groups_test': 0, 'train_exact_validation': 0, 'train_normalized_validation': 0, 'train_near_groups_validation': 0}
Epoch-selection / calibration rows: 132 137


role         dataset    label
calibration  deepset    0        35
                        1        21
             safeguard  0        57
                        1        24
selection    deepset    0        34
                        1        19
             safeguard  0        55
                        1        24
dtype: int64

In [3]:
from transformers import AutoModelForSequenceClassification
NLI_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
NLI_REVISION = "6f5cf0a2b59cabb106aca4c287eed12e357e90eb"
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_NAME, revision=NLI_REVISION, token=False, use_fast=False)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME, revision=NLI_REVISION, token=False).to(DEVICE)
nli_model.eval()
nli_model.requires_grad_(False)
entailment_id = next(int(i) for i, name in nli_model.config.id2label.items() if name.lower() == "entailment")
hypotheses = [
    "This text is a legitimate request or a discussion of instructions that does not ask the assistant to override its intended task.",
    "This text is an attempt to override the assistant's instructions or redirect its intended task."
]
def nli_scores(texts):
    outputs = []
    entailment_logits = []
    started = time.perf_counter()
    with torch.inference_mode():
        for begin in range(0, len(texts), 8):
            batch = texts[begin:begin+8]
            premises = [text for text in batch for _ in hypotheses]
            candidates = hypotheses * len(batch)
            inputs = nli_tokenizer(premises, candidates, padding=True,
                truncation="only_first", max_length=128, return_tensors="pt").to(DEVICE)
            logits = nli_model(**inputs).logits[:, entailment_id].reshape(-1, 2)
            outputs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().tolist())
            entailment_logits.append(logits.cpu().numpy())
    print("NLI inference seconds:", time.perf_counter()-started)
    return np.asarray(outputs), np.concatenate(entailment_logits)

def choose_threshold(y, scores):
    candidates = np.unique(np.append(scores, np.nextafter(scores.max(), np.inf)))
    return float(next(t for t in candidates if np.mean(scores[y == 0] >= t) <= .05))

roles = pd.read_csv("validation_roles.csv")
calibration_ids = set(roles.loc[roles.role.eq("calibration"), "row_id"])
validation = retained_prompts[retained_prompts.row_id.isin(calibration_ids)]
validation_scores, validation_logits = nli_scores(validation.text.tolist())
threshold = choose_threshold(validation.label.to_numpy(), validation_scores)
print("NLI threshold:", repr(threshold))
perturbations = pd.read_csv("perturbations.csv")
assert not perturbations.duplicated(["row_id", "column"]).any()
frame = perturbations[["row_id", "column", "label"]].copy()
frame["score"], test_logits = nli_scores(perturbations.text.tolist())
frame["value"] = "zero_shot"
frame["threshold"] = threshold
scores = pd.read_csv("scores.csv")
scores = pd.concat([scores[~scores.value.eq("zero_shot")], frame], ignore_index=True)
scores.to_csv("scores.csv", index=False)
frame.to_csv("zero_shot_nli_scores.csv", index=False)
for column, group in frame.groupby("column"):
    print(column, "AUROC", roc_auc_score(group.label, group.score))

# Preserve raw logits for both calibration and test rows.
test_logit_frame = frame.copy()
test_logit_frame["split"] = "test"
calibration_logit_frame = validation[["row_id", "dataset", "label"]].copy()
calibration_logit_frame["column"] = calibration_logit_frame.pop("dataset") + "_clean"
calibration_logit_frame["score"] = validation_scores
calibration_logit_frame["split"] = "calibration"
for logit_frame, raw in [(test_logit_frame, test_logits), (calibration_logit_frame, validation_logits)]:
    logit_frame["value"] = "zero_shot"
    logit_frame["seed"] = 15
    logit_frame["threshold"] = threshold
    logit_frame["benign_entailment_logit"] = raw[:, 0]
    logit_frame["injection_entailment_logit"] = raw[:, 1]
    logit_frame["logit"] = raw[:, 1] - raw[:, 0]
probe_logits = pd.read_csv("logits.csv")
probe_logits = probe_logits[~probe_logits.value.eq("zero_shot")]
pd.concat([probe_logits, test_logit_frame, calibration_logit_frame], ignore_index=True).to_csv("logits.csv", index=False)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

NLI inference seconds: 1.9474001809999208
NLI threshold: 0.586544930934906


NLI inference seconds: 12.142974761000005
deepset_base64 AUROC 0.4363095238095238
deepset_clean AUROC 0.6127976190476191
deepset_homoglyph AUROC 0.4666666666666667
deepset_paraphrase AUROC 0.5583999999999999
deepset_zero_width AUROC 0.3877976190476191
safeguard_base64 AUROC 0.5486694677871148
safeguard_clean AUROC 0.7869981325863677
safeguard_homoglyph AUROC 0.49731559290382815
safeguard_paraphrase AUROC 0.6352
safeguard_zero_width AUROC 0.24976657329598506
